# 07. Consultas heterogéneas con LLM local corregida

Esta libreta responde las 20 preguntas del proyecto usando SQLite, NetworkX, TF-IDF y un LLM local con Ollama.

Cambios principales de esta versión:

- Verifica que Ollama esté activo.
- Lista los modelos instalados y valida que exista `qwen2.5:7b`.
- Ejecuta una prueba real de generación antes de responder preguntas.
- Exige que las 20 respuestas usen el LLM local.
- Optimiza el prompt para respuestas más completas, claras y defendibles.
- Guarda respuestas, trazabilidad y diagnóstico de uso del LLM.

No usa API_KEY. Si el LLM local no está disponible, la libreta se detiene en vez de producir respuestas fallback.

## 1. Dependencias

In [ ]:
# Dependencias principales.
from pathlib import Path
from itertools import combinations
from difflib import SequenceMatcher
import re
import sqlite3
import warnings
import json
import time

import joblib
import networkx as nx
import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

## 2. Rutas del proyecto

In [ ]:
# Rutas del proyecto.
RAIZ_PROYECTO = Path(r"C:\Users\hazar\Desktop\PROYECTO")

DIR_DATOS = RAIZ_PROYECTO / "06_LLM" / "01_Datos"
DIR_GRAFOS = RAIZ_PROYECTO / "06_LLM" / "00_Grafos"
DIR_CONSULTAS = RAIZ_PROYECTO / "06_LLM" / "02_Consultas"
DIR_CONSULTAS.mkdir(parents=True, exist_ok=True)

RUTA_SQLITE = DIR_DATOS / "UNAM_Completo_2024_2025.sqlite"
RUTA_TFIDF_ARTICULOS = DIR_DATOS / "tfidf_articulos.joblib"
RUTA_TFIDF_AUTORES = DIR_DATOS / "tfidf_autores.joblib"
RUTA_GRAFO = DIR_GRAFOS / "BD_UNAM_Graph_LLM.gml"
RUTA_SUBGRAFO = DIR_GRAFOS / "BD_UNAM_Sub_Graph_LLM.gml"
RUTA_PREGUNTAS = DIR_CONSULTAS / "preguntas_consultas_heterogeneas_simple.csv"


FALLBACKS = {
    str(RUTA_SQLITE): Path("/mnt/data/UNAM_Completo_2024_2025.sqlite"),
    str(RUTA_TFIDF_ARTICULOS): Path("/mnt/data/tfidf_articulos.joblib"),
    str(RUTA_TFIDF_AUTORES): Path("/mnt/data/tfidf_autores.joblib"),
    str(RUTA_GRAFO): Path("/mnt/data/BD_UNAM_Graph_LLM.gml"),
    str(RUTA_SUBGRAFO): Path("/mnt/data/BD_UNAM_Sub_Graph_LLM.gml"),
    str(RUTA_PREGUNTAS): Path("/mnt/data/preguntas_consultas_heterogeneas_simple.csv"),
}

def resolver_ruta(ruta):
    ruta = Path(ruta)
    if ruta.exists():
        return ruta
    fb = FALLBACKS.get(str(ruta))
    if fb and fb.exists():
        return fb
    raise FileNotFoundError(f"No se encontró el archivo: {ruta}")

RUTA_SQLITE = resolver_ruta(RUTA_SQLITE)
RUTA_TFIDF_ARTICULOS = resolver_ruta(RUTA_TFIDF_ARTICULOS)
RUTA_TFIDF_AUTORES = resolver_ruta(RUTA_TFIDF_AUTORES)
RUTA_GRAFO = resolver_ruta(RUTA_GRAFO)
RUTA_SUBGRAFO = resolver_ruta(RUTA_SUBGRAFO)
RUTA_PREGUNTAS = resolver_ruta(RUTA_PREGUNTAS)

print("SQLite:", RUTA_SQLITE)
print("TF-IDF artículos:", RUTA_TFIDF_ARTICULOS)
print("TF-IDF autores:", RUTA_TFIDF_AUTORES)
print("Grafo principal:", RUTA_GRAFO)
print("Subgrafo:", RUTA_SUBGRAFO)
print("Preguntas:", RUTA_PREGUNTAS)

SQLite: C:\Users\hazar\Desktop\PROYECTO\06_LLM\01_Datos\UNAM_Completo_2024_2025.sqlite
TF-IDF artículos: C:\Users\hazar\Desktop\PROYECTO\06_LLM\01_Datos\tfidf_articulos.joblib
TF-IDF autores: C:\Users\hazar\Desktop\PROYECTO\06_LLM\01_Datos\tfidf_autores.joblib
Grafo principal: C:\Users\hazar\Desktop\PROYECTO\06_LLM\00_Grafos\BD_UNAM_Graph_LLM.gml
Subgrafo: C:\Users\hazar\Desktop\PROYECTO\06_LLM\00_Grafos\BD_UNAM_Sub_Graph_LLM.gml
Preguntas: C:\Users\hazar\Desktop\PROYECTO\06_LLM\02_Consultas\preguntas_consultas_heterogeneas_simple.csv


## 3. Carga de recursos: SQLite, grafos, TF-IDF y preguntas

In [3]:
# Cargar SQLite.
conn = sqlite3.connect(RUTA_SQLITE)
df = pd.read_sql_query("SELECT * FROM articulos_autores", conn)

# Normalización mínima de tipos.
df["indice"] = pd.to_numeric(df["indice"], errors="coerce").astype("Int64")
df["Año"] = pd.to_numeric(df["Año"], errors="coerce").astype("Int64")
for col in df.columns:
    if col not in ["indice", "Año"]:
        df[col] = df[col].fillna("").astype(str).str.strip()

# Cargar grafos.
G = nx.read_gml(RUTA_GRAFO)
G_sub = nx.read_gml(RUTA_SUBGRAFO)

# Cargar TF-IDF.
tfidf_articulos = joblib.load(RUTA_TFIDF_ARTICULOS)
tfidf_autores = joblib.load(RUTA_TFIDF_AUTORES)

vectorizer_articulos = tfidf_articulos["vectorizer"]
matriz_articulos = tfidf_articulos["matrix"]
meta_articulos = tfidf_articulos["metadata"].copy()

vectorizer_autores = tfidf_autores["vectorizer"]
matriz_autores = tfidf_autores["matrix"]
meta_autores = tfidf_autores["metadata"].copy()

# Cargar preguntas.
preguntas = pd.read_csv(RUTA_PREGUNTAS)

print("Filas base:", len(df))
print("Artículos:", df["indice"].nunique())
print("Autores:", df["Autor_norm"].nunique())
print("Grafo principal:", G.number_of_nodes(), "nodos /", G.number_of_edges(), "aristas")
print("Subgrafo:", G_sub.number_of_nodes(), "nodos /", G_sub.number_of_edges(), "aristas")
print("Preguntas:", len(preguntas))

display(preguntas)

Filas base: 905
Artículos: 406
Autores: 550
Grafo principal: 3896 nodos / 5078 aristas
Subgrafo: 550 nodos / 827 aristas
Preguntas: 20


,id,tipo,pregunta,componentes_consultados
0,D1,descriptiva,¿Qué áreas de computación concentran más artículos por periodo de publicación?,SQL + grafo principal
1,D2,descriptiva,¿Qué autores tienen mayor producción dentro de cada área de conocimiento?,SQL + grafo principal
2,D3,descriptiva,¿Qué artículos tienen mayor número de coautores y a qué afiliaciones pertenecen esos autores?,SQL + grafo principal
3,D4,descriptiva,¿Qué keywords aparecen con mayor frecuencia y en qué áreas se concentran?,SQL + TF-IDF + grafo principal
4,D5,descriptiva,"Para la keyword image processing, ¿qué afiliaciones concentran más autores y artículos relacionados con ese tema?",SQL + TF-IDF + grafo principal
5,D6,descriptiva,¿Qué autores conectan diferentes áreas o afiliaciones mediante sus publicaciones y coautorías?,SQL + grafo principal + subgrafo de coautoría
6,D7,descriptiva,"Para la keyword brain, ¿qué artículos la utilizan y qué autores participaron en ellos?",TF-IDF + grafo principal
7,D8,descriptiva,¿Qué áreas tienen producción reciente pero pocos autores asociados?,SQL + grafo principal
8,D9,descriptiva,"¿Qué autores trabajan temas similares, según keywords y abstracts, aunque todavía no aparezcan como coautores?",TF-IDF + grafo principal + subgrafo de coautoría
9,D10,descriptiva,"¿Qué publicaciones parecen duplicadas o variantes del mismo artículo al comparar título, DOI, URL, abstract y autores?",SQL + TF-IDF


## 4. Funciones auxiliares de texto, SQL, NetworkX y TF-IDF

In [4]:
def texto_limpio(valor):
    if pd.isna(valor):
        return ""
    return str(valor).strip()


def normalizar_basico(texto):
    texto = texto_limpio(texto).lower()
    texto = texto.replace("ϵ", "epsilon").replace("ε", "epsilon")
    texto = re.sub(r"[^a-z0-9áéíóúñü]+", " ", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def separar_keywords(valor):
    texto = texto_limpio(valor).lower()
    if not texto:
        return []
    partes = re.split(r"[,;|]", texto)
    partes = [re.sub(r"\s+", " ", p).strip(" .:;,-") for p in partes]
    return sorted({p for p in partes if p})


def tabla_md(tabla, max_filas=12):
    if tabla is None:
        return ""
    if isinstance(tabla, pd.Series):
        tabla = tabla.reset_index()
    if not isinstance(tabla, pd.DataFrame):
        return str(tabla)
    if tabla.empty:
        return "Sin resultados."
    return tabla.head(max_filas).to_markdown(index=False)


def ejecutar_sql(sql):
    return pd.read_sql_query(sql, conn)

# Perfiles precomputados para evitar recorrer toda la base miles de veces.
AUTHOR_INFO = {}
KEYWORD_AUTORES = {}
for autor, sub in df.groupby("Autor_norm"):
    kws = set()
    for val in sub["Keywords"].dropna().unique():
        kws.update(separar_keywords(val))
    areas = set(sub["Area"].dropna().astype(str)) - {""}
    anios = set(sub["Año"].dropna().astype(int))
    affs = (set(sub["Afiliacion1"].dropna().astype(str)) | set(sub["Afiliacion2"].dropna().astype(str))) - {""}
    AUTHOR_INFO[autor] = {
        "keywords": kws,
        "areas": areas,
        "anios": anios,
        "afiliaciones": affs,
        "grado": G_sub.degree(autor) if autor in G_sub else 0,
        "articulos": set(sub["indice"].dropna().astype(int))
    }
    for kw in kws:
        KEYWORD_AUTORES.setdefault(kw, set()).add(autor)

AUTORES = sorted(AUTHOR_INFO.keys())
_SIM_AUTORES = None


def matriz_sim_autores():
    global _SIM_AUTORES
    if _SIM_AUTORES is None:
        _SIM_AUTORES = cosine_similarity(matriz_autores)
    return _SIM_AUTORES


def areas_de_autor(autor):
    return sorted(AUTHOR_INFO.get(autor, {}).get("areas", set()))


def anios_de_autor(autor):
    return sorted(AUTHOR_INFO.get(autor, {}).get("anios", set()))


def afiliaciones_de_autor(autor):
    return sorted(AUTHOR_INFO.get(autor, {}).get("afiliaciones", set()))


def keywords_de_autor(autor):
    return sorted(AUTHOR_INFO.get(autor, {}).get("keywords", set()))


def conectados(autor1, autor2):
    return autor1 in G_sub and autor2 in G_sub and G_sub.has_edge(autor1, autor2)


def buscar_tfidf_articulos(consulta, top_k=10):
    q = vectorizer_articulos.transform([consulta])
    sims = cosine_similarity(q, matriz_articulos).ravel()
    idx = np.argsort(sims)[::-1][:top_k]
    res = meta_articulos.iloc[idx].copy()
    res.insert(0, "similitud", sims[idx])
    return res[res["similitud"] > 0].reset_index(drop=True)


def buscar_tfidf_autores(consulta, top_k=10):
    q = vectorizer_autores.transform([consulta])
    sims = cosine_similarity(q, matriz_autores).ravel()
    idx = np.argsort(sims)[::-1][:top_k]
    res = meta_autores.iloc[idx].copy()
    res.insert(0, "similitud", sims[idx])
    return res[res["similitud"] > 0].reset_index(drop=True)


def similitud_autores_top(no_conectados=True, distinta_area=None, misma_afiliacion=None, distinta_afiliacion=None, top_k=15):
    sims = matriz_sim_autores()
    autores = meta_autores["Autor_norm"].tolist()
    registros = []
    for i, j in combinations(range(len(autores)), 2):
        a1, a2 = autores[i], autores[j]
        sim = float(sims[i, j])
        if sim <= 0:
            continue
        if no_conectados and conectados(a1, a2):
            continue
        areas1, areas2 = set(areas_de_autor(a1)), set(areas_de_autor(a2))
        aff1, aff2 = set(afiliaciones_de_autor(a1)), set(afiliaciones_de_autor(a2))
        if distinta_area is True and areas1.intersection(areas2):
            continue
        if misma_afiliacion is True and not aff1.intersection(aff2):
            continue
        if distinta_afiliacion is True and aff1.intersection(aff2):
            continue
        registros.append({
            "autor_1": a1,
            "autor_2": a2,
            "similitud_tfidf": round(sim, 4),
            "areas_1": "; ".join(sorted(areas1)),
            "areas_2": "; ".join(sorted(areas2)),
            "afiliaciones_compartidas": "; ".join(sorted(aff1.intersection(aff2))),
            "grado_1": AUTHOR_INFO.get(a1, {}).get("grado", 0),
            "grado_2": AUTHOR_INFO.get(a2, {}).get("grado", 0),
        })
    if not registros:
        return pd.DataFrame()
    return pd.DataFrame(registros).sort_values("similitud_tfidf", ascending=False).head(top_k).reset_index(drop=True)


def pares_por_keywords(filtro_autores=None, min_keywords=2, top_k=15, max_autores_por_keyword=120):
    filtro = set(filtro_autores) if filtro_autores is not None else set(AUTORES)
    pares = {}
    for kw, autores_kw in KEYWORD_AUTORES.items():
        autores_validos = sorted(autores_kw & filtro)
        # Evita keywords demasiado genéricas que generan muchos pares poco útiles.
        if len(autores_validos) < 2 or len(autores_validos) > max_autores_por_keyword:
            continue
        for a1, a2 in combinations(autores_validos, 2):
            if conectados(a1, a2):
                continue
            key = tuple(sorted((a1, a2)))
            pares.setdefault(key, set()).add(kw)
    registros = []
    for (a1, a2), kws in pares.items():
        if len(kws) >= min_keywords:
            registros.append({
                "autor_1": a1,
                "autor_2": a2,
                "keywords_compartidas": len(kws),
                "keywords": "; ".join(sorted(kws)[:8]),
                "grado_1": AUTHOR_INFO.get(a1, {}).get("grado", 0),
                "grado_2": AUTHOR_INFO.get(a2, {}).get("grado", 0),
            })
    if not registros:
        return pd.DataFrame()
    return pd.DataFrame(registros).sort_values("keywords_compartidas", ascending=False).head(top_k).reset_index(drop=True)


def resumen_resultado_para_llm(resultado):
    partes = []
    for k, v in resultado.items():
        if isinstance(v, pd.DataFrame):
            partes.append(f"\n[{k}]\n{tabla_md(v, 12)}")
        elif isinstance(v, (list, tuple)):
            partes.append(f"\n[{k}]\n" + "\n".join(map(str, v[:20])))
        elif k not in {"codigo_sql", "codigo_networkx", "consulta_tfidf"}:
            partes.append(f"\n[{k}]\n{v}")
    return "\n".join(partes)

## 5. Resolutores de las preguntas descriptivas y predictivas

In [5]:
def resolver_D1():
    sql = """
    SELECT Año, Area, COUNT(DISTINCT indice) AS articulos
    FROM articulos_autores
    GROUP BY Año, Area
    ORDER BY Año, articulos DESC;
    """
    return {"motores": "SQL + grafo principal", "codigo_sql": sql.strip(), "resultado": ejecutar_sql(sql)}


def resolver_D2():
    conteo = (df.groupby(["Area", "Autor_norm"])["indice"].nunique()
                .reset_index(name="articulos")
                .sort_values(["Area", "articulos", "Autor_norm"], ascending=[True, False, True]))
    tabla = conteo.groupby("Area", group_keys=False).head(5).reset_index(drop=True)
    return {"motores": "SQL + grafo principal", "codigo_sql": "GROUP BY Area, Autor_norm; top 5 por área", "resultado": tabla}


def resolver_D3():
    base = df.groupby(["indice", "Titulo", "Area", "Año"]).agg(
        total_autores=("Autor_norm", "nunique"),
        autores=("Autor_norm", lambda x: "; ".join(sorted(set(x)))),
        afiliaciones=("Afiliacion1", lambda x: "; ".join(sorted({v for v in x if str(v).strip()})))
    ).reset_index()
    tabla = base.sort_values("total_autores", ascending=False).head(12)
    return {"motores": "SQL + grafo principal", "codigo_sql": "COUNT(DISTINCT Autor_norm) por artículo", "resultado": tabla}


def resolver_D4():
    registros = []
    for _, r in df[["indice", "Area", "Keywords"]].drop_duplicates().iterrows():
        for kw in separar_keywords(r["Keywords"]):
            registros.append({"keyword": kw, "Area": r["Area"], "indice": int(r["indice"])})
    kdf = pd.DataFrame(registros)
    if kdf.empty:
        return {"motores": "SQL + TF-IDF + grafo principal", "resultado": pd.DataFrame()}
    freq = kdf.groupby("keyword")["indice"].nunique().reset_index(name="articulos").sort_values("articulos", ascending=False).head(20)
    area_kw = kdf.groupby(["keyword", "Area"])["indice"].nunique().reset_index(name="articulos")
    area_top = area_kw.sort_values(["keyword", "articulos"], ascending=[True, False]).groupby("keyword").head(1)
    tabla = freq.merge(area_top[["keyword", "Area"]], on="keyword", how="left").rename(columns={"Area": "area_principal"})
    return {"motores": "SQL + TF-IDF + grafo principal", "consulta_tfidf": "keywords frecuentes", "resultado": tabla}


def resolver_D5():
    consulta = "image processing"
    tf = buscar_tfidf_articulos(consulta, top_k=40)
    indices_tfidf = set(tf["indice"].astype(int).tolist()) if not tf.empty else set()
    indices_exactos = set()
    for idx, kws in df[["indice", "Keywords"]].drop_duplicates().values:
        if any(consulta in kw for kw in separar_keywords(kws)):
            indices_exactos.add(int(idx))
    indices = indices_exactos | set(list(indices_tfidf)[:20])
    sub = df[df["indice"].astype(int).isin(indices)].copy()
    tabla = sub.groupby("Afiliacion1").agg(articulos=("indice", "nunique"), autores=("Autor_norm", "nunique")).reset_index()
    tabla = tabla.sort_values(["articulos", "autores"], ascending=False).head(15)
    return {"motores": "SQL + TF-IDF + grafo principal", "consulta_tfidf": consulta, "resultado": tabla, "articulos_relacionados": tf[["indice", "Titulo", "similitud"]].head(10)}


def resolver_D6():
    perfiles = []
    for autor, info in AUTHOR_INFO.items():
        areas = sorted(info["areas"])
        affs = sorted(info["afiliaciones"])
        grado = info["grado"]
        perfiles.append({"Autor_norm": autor, "articulos": len(info["articulos"]), "num_areas": len(areas), "num_afiliaciones": len(affs), "grado_coautoria": grado, "areas": "; ".join(areas), "afiliaciones": "; ".join(affs[:4]), "puntaje": len(areas)*2 + len(affs) + grado})
    tabla = pd.DataFrame(perfiles).sort_values("puntaje", ascending=False).head(15)
    return {"motores": "SQL + grafo principal + subgrafo de coautoría", "codigo_networkx": "grado en subgrafo + diversidad de áreas/afiliaciones", "resultado": tabla.drop(columns=["puntaje"])}


def resolver_D7():
    consulta = "brain"
    tf = buscar_tfidf_articulos(consulta, top_k=30)
    indices_exactos = set()
    for idx, kws in df[["indice", "Keywords"]].drop_duplicates().values:
        if any("brain" in kw for kw in separar_keywords(kws)):
            indices_exactos.add(int(idx))
    indices = indices_exactos | set(tf["indice"].astype(int).head(15).tolist())
    sub = df[df["indice"].astype(int).isin(indices)]
    tabla = sub.groupby(["indice", "Titulo", "Area", "Año"]).agg(autores=("Autor_norm", lambda x: "; ".join(sorted(set(x))))).reset_index().head(20)
    return {"motores": "TF-IDF + grafo principal", "consulta_tfidf": consulta, "resultado": tabla}


def resolver_D8():
    anio_reciente = int(df["Año"].dropna().max())
    sub = df[df["Año"].astype("Int64") == anio_reciente]
    tabla = sub.groupby("Area").agg(articulos=("indice", "nunique"), autores=("Autor_norm", "nunique")).reset_index()
    tabla["articulos_por_autor"] = (tabla["articulos"] / tabla["autores"].replace(0, np.nan)).round(3)
    tabla = tabla.sort_values(["autores", "articulos"], ascending=[True, False])
    return {"motores": "SQL + grafo principal", "resultado": tabla, "nota": f"Se tomó como producción reciente el año {anio_reciente}."}


def resolver_D9():
    return {"motores": "TF-IDF + grafo principal + subgrafo de coautoría", "consulta_tfidf": "similitud entre perfiles textuales de autores", "resultado": similitud_autores_top(no_conectados=True, top_k=15)}


def resolver_D10():
    arts = meta_articulos.copy().reset_index(drop=True)
    registros = []

    # 1) Coincidencias exactas por DOI o URL no vacíos.
    for campo, etiqueta in [("Doi", "mismo DOI"), ("URL", "misma URL")]:
        serie = arts[campo].fillna("").astype(str).str.strip().str.lower()
        for valor, idxs in serie[serie.ne("")].groupby(serie).groups.items():
            idxs = list(idxs)
            if len(idxs) > 1:
                for i, j in combinations(idxs, 2):
                    registros.append({
                        "indice_1": arts.loc[i, "indice"], "titulo_1": arts.loc[i, "Titulo"],
                        "indice_2": arts.loc[j, "indice"], "titulo_2": arts.loc[j, "Titulo"],
                        "sim_titulo": 1.0 if normalizar_basico(arts.loc[i, "Titulo"]) == normalizar_basico(arts.loc[j, "Titulo"]) else np.nan,
                        "sim_tfidf": np.nan,
                        "razon": etiqueta
                    })

    # 2) Sólo revisar pares con similitud textual alta para evitar comparar todos contra todos.
    sims = cosine_similarity(matriz_articulos)
    tri = np.triu_indices_from(sims, k=1)
    valores = sims[tri]
    candidatos = np.where(valores >= 0.45)[0]
    if len(candidatos) > 300:
        candidatos = candidatos[np.argsort(valores[candidatos])[::-1][:300]]

    for pos in candidatos:
        i, j = tri[0][pos], tri[1][pos]
        t1, t2 = str(arts.loc[i, "Titulo"]), str(arts.loc[j, "Titulo"])
        ratio_titulo = SequenceMatcher(None, normalizar_basico(t1), normalizar_basico(t2)).ratio()
        sim_texto = float(sims[i, j])
        razon = []
        if ratio_titulo >= 0.88:
            razon.append("título muy similar")
        if sim_texto >= 0.70:
            razon.append("texto muy similar")
        if razon:
            registros.append({
                "indice_1": arts.loc[i, "indice"], "titulo_1": t1,
                "indice_2": arts.loc[j, "indice"], "titulo_2": t2,
                "sim_titulo": round(ratio_titulo, 3),
                "sim_tfidf": round(sim_texto, 3),
                "razon": "; ".join(razon)
            })

    tabla = pd.DataFrame(registros).drop_duplicates() if registros else pd.DataFrame()
    if not tabla.empty:
        tabla = tabla.sort_values(["sim_titulo", "sim_tfidf"], ascending=False, na_position="last").head(20)
    return {"motores": "SQL + TF-IDF", "consulta_tfidf": "detección de variantes/duplicados", "resultado": tabla}

def resolver_P1():
    tabla = pares_por_keywords(min_keywords=2, top_k=15)
    return {"motores": "TF-IDF/keywords + grafo principal", "resultado": tabla}


def resolver_P2():
    bajos = {a for a, info in AUTHOR_INFO.items() if info["grado"] <= 1}
    sim = similitud_autores_top(no_conectados=True, top_k=200)
    tabla = sim[sim["autor_1"].isin(bajos) | sim["autor_2"].isin(bajos)].head(15)
    return {"motores": "grafo principal + subgrafo de coautoría", "resultado": tabla}


def resolver_P3():
    registros = []
    for a1, a2 in combinations(list(G_sub.nodes()), 2):
        if G_sub.has_edge(a1, a2): continue
        cn = len(list(nx.common_neighbors(G_sub, a1, a2)))
        if cn == 0: continue
        union = len(set(G_sub.neighbors(a1)) | set(G_sub.neighbors(a2)))
        jaccard = cn / union if union else 0
        score = cn + jaccard + np.log1p(G_sub.degree(a1) + G_sub.degree(a2)) / 10
        registros.append({"autor_1": a1, "autor_2": a2, "vecinos_comunes": cn, "jaccard": round(jaccard, 3), "puntaje": round(score, 3)})
    tabla = pd.DataFrame(registros).sort_values("puntaje", ascending=False).head(15) if registros else pd.DataFrame()
    return {"motores": "subgrafo de coautoría", "codigo_networkx": "common_neighbors + jaccard sobre no-aristas", "resultado": tabla}


def resolver_P4():
    consulta = "image processing"
    arts = buscar_tfidf_articulos(consulta, top_k=40)
    indices = set(arts["indice"].astype(int).tolist())
    autores = set(df[df["indice"].astype(int).isin(indices)]["Autor_norm"].unique())
    tabla = pares_por_keywords(filtro_autores=autores, min_keywords=1, top_k=15)
    return {"motores": "TF-IDF + grafo principal + subgrafo de coautoría", "consulta_tfidf": consulta, "resultado": tabla}


def resolver_P5():
    candidatos = pares_por_keywords(min_keywords=1, top_k=500)
    if candidatos.empty:
        return {"motores": "SQL + TF-IDF + subgrafo de coautoría", "resultado": pd.DataFrame()}
    registros = []
    for _, r in candidatos.iterrows():
        a1, a2 = r["autor_1"], r["autor_2"]
        aff = set(afiliaciones_de_autor(a1)) & set(afiliaciones_de_autor(a2))
        if aff:
            reg = r.to_dict()
            reg["afiliacion_compartida"] = "; ".join(sorted(aff)[:2])
            registros.append(reg)
    tabla = pd.DataFrame(registros).sort_values("keywords_compartidas", ascending=False).head(15) if registros else pd.DataFrame()
    return {"motores": "SQL + TF-IDF + subgrafo de coautoría", "resultado": tabla}


def resolver_P6():
    return {"motores": "SQL + TF-IDF + subgrafo de coautoría", "resultado": similitud_autores_top(no_conectados=True, distinta_afiliacion=True, top_k=15)}


def resolver_P7():
    registros = []
    for a, info in AUTHOR_INFO.items():
        if info["grado"] > 1:
            continue
        areas_a = info["areas"]
        posibles = []
        for b, info_b in AUTHOR_INFO.items():
            if a == b or conectados(a, b): continue
            inter_area = areas_a.intersection(info_b["areas"])
            if inter_area:
                posibles.append((b, info_b["grado"], "; ".join(sorted(inter_area))))
        for b, grado, area in sorted(posibles, key=lambda x: x[1], reverse=True)[:3]:
            registros.append({"autor_poco_conectado": a, "grado_actual": info["grado"], "autor_grupo_sugerido": b, "grado_grupo": grado, "area_compartida": area})
    return {"motores": "grafo principal + subgrafo de coautoría", "resultado": pd.DataFrame(registros).head(15)}


def resolver_P8():
    registros = []
    for a1, a2 in combinations(AUTORES, 2):
        if conectados(a1, a2): continue
        areas = AUTHOR_INFO[a1]["areas"] & AUTHOR_INFO[a2]["areas"]
        anios = AUTHOR_INFO[a1]["anios"] & AUTHOR_INFO[a2]["anios"]
        if not areas or not anios: continue
        inter = AUTHOR_INFO[a1]["keywords"] & AUTHOR_INFO[a2]["keywords"]
        registros.append({"autor_1": a1, "autor_2": a2, "años_compartidos": "; ".join(map(str, sorted(anios))), "areas_compartidas": "; ".join(sorted(areas)), "keywords_compartidas": len(inter)})
    tabla = pd.DataFrame(registros).sort_values("keywords_compartidas", ascending=False).head(15) if registros else pd.DataFrame()
    return {"motores": "SQL + subgrafo de coautoría", "resultado": tabla}


def resolver_P9():
    aff_registros = []
    for autor, info in AUTHOR_INFO.items():
        for aff in info["afiliaciones"]:
            aff_registros.append({"Autor_norm": autor, "afiliacion": aff})
    aff_df = pd.DataFrame(aff_registros)
    if aff_df.empty:
        return {"motores": "SQL + grafo principal + subgrafo de coautoría", "resultado": pd.DataFrame()}
    top_aff = aff_df.groupby("afiliacion")["Autor_norm"].nunique().sort_values(ascending=False).index[0]
    autores_aff = set(aff_df[aff_df["afiliacion"].eq(top_aff)]["Autor_norm"].unique())
    candidatos = pares_por_keywords(filtro_autores=autores_aff, min_keywords=0, top_k=500)
    registros = []
    # Si no hay keywords compartidas, revisar pares por área dentro de la afiliación.
    pares_base = list(combinations(sorted(autores_aff), 2)) if candidatos.empty else [(r.autor_1, r.autor_2) for r in candidatos.itertuples()]
    for a1, a2 in pares_base:
        if conectados(a1, a2): continue
        areas = AUTHOR_INFO[a1]["areas"] & AUTHOR_INFO[a2]["areas"]
        if not areas: continue
        inter = AUTHOR_INFO[a1]["keywords"] & AUTHOR_INFO[a2]["keywords"]
        registros.append({"autor_1": a1, "autor_2": a2, "afiliacion": top_aff, "areas_compartidas": "; ".join(sorted(areas)), "keywords_compartidas": len(inter)})
    tabla = pd.DataFrame(registros).sort_values("keywords_compartidas", ascending=False).head(15) if registros else pd.DataFrame()
    return {"motores": "SQL + grafo principal + subgrafo de coautoría", "resultado": tabla, "afiliacion_mayor": top_aff}


def resolver_P10():
    return {"motores": "SQL + TF-IDF + subgrafo de coautoría", "resultado": similitud_autores_top(no_conectados=True, distinta_area=True, top_k=15)}


RESOLUTORES = {
    "D1": resolver_D1, "D2": resolver_D2, "D3": resolver_D3, "D4": resolver_D4, "D5": resolver_D5,
    "D6": resolver_D6, "D7": resolver_D7, "D8": resolver_D8, "D9": resolver_D9, "D10": resolver_D10,
    "P1": resolver_P1, "P2": resolver_P2, "P3": resolver_P3, "P4": resolver_P4, "P5": resolver_P5,
    "P6": resolver_P6, "P7": resolver_P7, "P8": resolver_P8, "P9": resolver_P9, "P10": resolver_P10,
}

## 6. Configuración estricta de Ollama y prompt optimizado

In [ ]:
# ============================================================
# Configuración del LLM local con Ollama
# ============================================================

USAR_LLM = True
REQUERIR_LLM_EN_TODAS = True

MODELO_OLLAMA = "qwen2.5:7b"

OLLAMA_BASE_URLS = [
    "http://localhost:11434",
    "http://127.0.0.1:11434",
]

OLLAMA_URL = None
LLM_DISPONIBLE = False

LLM_INFO = {
    "modelo_configurado": MODELO_OLLAMA,
    "endpoint": None,
    "modelos_disponibles": [],
    "prueba_respuesta": "",
}

# Códigos válidos de área.
CODIGOS_AREA_VALIDOS = ["ISBD", "CC", "IA", "TC", "SIAV", "RS"]


# ============================================================
# Detección robusta de Ollama
# ============================================================

def detectar_ollama(modelo=MODELO_OLLAMA, timeout_tags=10, timeout_generate=300):
    """
    Detecta Ollama, valida que el modelo exista y prueba una generación corta.
    Si REQUERIR_LLM_EN_TODAS=True, detiene la libreta si no hay LLM local.
    """
    if not USAR_LLM:
        return None, False

    ultimo_error = None

    for base_url in OLLAMA_BASE_URLS:
        try:
            # 1. Verificar conexión con Ollama
            r = requests.get(f"{base_url}/api/tags", timeout=timeout_tags)

            if r.status_code != 200:
                ultimo_error = f"{base_url}/api/tags respondió status {r.status_code}: {r.text[:500]}"
                continue

            data = r.json()
            modelos = [m.get("name", "") for m in data.get("models", [])]
            LLM_INFO["modelos_disponibles"] = modelos

            print(f"Ollama detectado en: {base_url}")
            print("Modelos disponibles:")
            for m in modelos:
                print(" -", m)

            # 2. Verificar que el modelo exista exactamente
            if modelo not in modelos:
                raise RuntimeError(
                    f"El modelo configurado '{modelo}' no aparece en Ollama.\n"
                    f"Modelos disponibles: {modelos}\n"
                    "Solución: cambia MODELO_OLLAMA por uno de esos nombres "
                    "o ejecuta en PowerShell: ollama pull qwen2.5:7b"
                )

            # 3. Probar generación real
            payload = {
                "model": modelo,
                "prompt": "Responde exactamente con la palabra OK.",
                "stream": False,
                "options": {
                    "temperature": 0.0,
                    "num_predict": 10,
                }
            }

            r2 = requests.post(
                f"{base_url}/api/generate",
                json=payload,
                timeout=timeout_generate
            )

            if r2.status_code != 200:
                raise RuntimeError(
                    f"Ollama respondió error en /api/generate. "
                    f"Status {r2.status_code}: {r2.text[:1000]}"
                )

            respuesta = r2.json().get("response", "").strip()

            if not respuesta:
                raise RuntimeError("Ollama respondió 200, pero la respuesta vino vacía.")

            LLM_INFO["endpoint"] = f"{base_url}/api/generate"
            LLM_INFO["prueba_respuesta"] = respuesta

            return f"{base_url}/api/generate", True

        except Exception as e:
            ultimo_error = repr(e)
            print(f"No se pudo usar {base_url}: {ultimo_error}")

    if REQUERIR_LLM_EN_TODAS:
        raise RuntimeError(
            "No se detectó Ollama/modelo local de forma válida.\n"
            "Verifica en PowerShell:\n"
            "  ollama list\n"
            "  ollama run qwen2.5:7b\n"
            "  ollama serve\n"
            f"Último error: {ultimo_error}"
        )

    return None, False


OLLAMA_URL, LLM_DISPONIBLE = detectar_ollama(MODELO_OLLAMA)

if LLM_DISPONIBLE:
    print(f"LLM local disponible: {MODELO_OLLAMA}")
    print(f"Endpoint usado: {OLLAMA_URL}")
    print(f"Prueba inicial del modelo: {LLM_INFO['prueba_respuesta']}")
else:
    print("LLM no disponible; se usará fallback sólo si REQUERIR_LLM_EN_TODAS=False.")


# ============================================================
# Llamada al LLM local
# ============================================================

def llamar_llm_ollama(prompt, modelo=MODELO_OLLAMA, timeout=600):
    """
    Llama al LLM local mediante Ollama.
    Devuelve texto si la llamada fue exitosa.
    Devuelve None si falla.
    """
    if not USAR_LLM or not LLM_DISPONIBLE or not OLLAMA_URL:
        return None

    payload = {
        "model": modelo,
        "prompt": prompt,
        "stream": False,
        "options": {
            # Temperatura baja para evitar creatividad excesiva.
            "temperature": 0.02,
            "top_p": 0.85,

            # Contexto amplio para que lea bien evidencia tabular.
            "num_ctx": 8192,

            # Suficiente para responder sin extenderse demasiado.
            "num_predict": 1100,
        }
    }

    try:
        inicio = time.time()
        r = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
        duracion = round(time.time() - inicio, 2)

        if r.status_code != 200:
            print("Error HTTP al llamar Ollama:", r.status_code)
            print(r.text[:1000])
            return None

        respuesta = r.json().get("response", "").strip()

        if not respuesta:
            print("Ollama respondió vacío.")
            return None

        return postprocesar_respuesta_llm(respuesta)

    except Exception as e:
        print("Error llamando al LLM local:")
        print(repr(e))
        return None


# ============================================================
# Postproceso defensivo de la respuesta
# ============================================================

def postprocesar_respuesta_llm(respuesta):
    """
    Limpieza ligera para evitar expansiones inventadas de códigos de área.
    No cambia datos ni agrega información; sólo evita frases problemáticas.
    """

    reemplazos = {
        "Inteligencia Artificial (IA)": "IA",
        "Ciencias de la Computación (CC)": "CC",
        "Teoría de la Computación (TC)": "TC",
        "Redes Sociales (RS)": "RS",

        # Variantes incorrectas que el LLM puede inventar para SIAV.
        "Sistemas e Investigación Aplicada a la Vida (SIAV)": "SIAV",
        "Sistemas e Investigación en Áreas de Vigencia (SIAV)": "SIAV",
        "Sistemas e Ingeniería de Aviación (SIAV)": "SIAV",
        "Sistemas de Información y Análisis Visual (SIAV)": "SIAV",

        # Variantes incorrectas para ISBD.
        "Ingeniería de Software y Bases de Datos (ISBD)": "ISBD",
        "Información, Sistemas y Bases de Datos (ISBD)": "ISBD",
    }

    for malo, bueno in reemplazos.items():
        respuesta = respuesta.replace(malo, bueno)

    return respuesta.strip()


# ============================================================
# Prompt optimizado para respuestas finales
# ============================================================

def construir_prompt_respuesta(id_pregunta, tipo, pregunta, componentes, resultado):
    """
    Construye el prompt que recibe el LLM.
    El LLM sólo redacta; los datos ya fueron calculados con SQL, NetworkX y/o TF-IDF.
    """

    evidencia = resumen_resultado_para_llm(resultado)

    tipo = str(tipo).strip().lower()
    id_pregunta = str(id_pregunta).strip()
    componentes = str(componentes).strip()

    es_predictiva = tipo == "predictiva" or id_pregunta.startswith("P")

    if es_predictiva:
        instruccion_tipo = """
La pregunta es predictiva o de recomendación.
NO presentes los resultados como certezas ni como predicciones garantizadas.
Usa expresiones como:
- candidatos preliminares
- pares sugeridos para revisión
- señales de posible colaboración
- resultados exploratorios

Debes explicar el criterio real usado:
- Si se usó subgrafo de coautoría: habla de cercanía estructural, vecinos comunes, grado, Jaccard o ausencia de arista previa.
- Si se usó TF-IDF: habla de similitud textual o similitud temática basada en Titulo, Keywords y Abstract.
- Si se usó SQL: habla de filtros por área, año, afiliación o autor.
"""
    else:
        instruccion_tipo = """
La pregunta es descriptiva.
Resume el patrón principal observado y destaca los valores más importantes.
No presentes el resultado como predicción.
"""

    prompt = f"""
Eres un asistente de análisis bibliográfico/académico para una base curada de artículos UNAM en Ciencias de la Computación.

Tu tarea es redactar una respuesta final en español usando EXCLUSIVAMENTE la evidencia calculada que se proporciona abajo.

REGLAS OBLIGATORIAS:
1. No inventes autores, artículos, afiliaciones, DOI, URL, áreas, keywords ni cantidades.
2. No uses conocimiento externo.
3. No menciones internet, búsquedas externas ni información no presente en la evidencia.
4. No calcules totales nuevos, porcentajes o sumas si no aparecen explícitamente en la evidencia.
5. Si un dato no aparece en la evidencia, no lo afirmes.
6. Usa los códigos de área exactamente como aparecen:
   ISBD, CC, IA, TC, SIAV, RS.
7. No expandas, traduzcas ni interpretes el significado de los códigos de área.
8. No digas que SIAV significa algo; sólo escribe SIAV.
9. No digas que ISBD, CC, IA, TC o RS significan algo; sólo usa el código.
10. Si los resultados son top-k, dilo en la nota de alcance.
11. Si los resultados son preliminares, dilo claramente.

ID de pregunta: {id_pregunta}
Tipo de pregunta: {tipo}
Pregunta original: {pregunta}
Componentes consultados según archivo de preguntas: {componentes}
Motores usados realmente: {resultado.get("motores", componentes)}

EVIDENCIA CALCULADA POR SQL / NETWORKX / TF-IDF:
{evidencia}

INSTRUCCIONES SEGÚN TIPO DE PREGUNTA:
{instruccion_tipo}

ESTRUCTURA OBLIGATORIA DE LA RESPUESTA:
1. Respuesta directa:
   - Contesta la pregunta en 2 a 5 oraciones.
   - Usa sólo datos de la evidencia.
   - Si mencionas áreas, usa sólo códigos.

2. Evidencia principal:
   - Lista los resultados más importantes.
   - Incluye cantidades, autores, artículos, áreas o afiliaciones sólo cuando aparezcan en la evidencia.

3. Criterio usado:
   - Explica brevemente qué componente permitió responder.
   - SQL: conteos, agrupaciones o filtros.
   - Grafo principal: relaciones artículo-autor-keyword-afiliación.
   - Subgrafo de coautoría: coautorías, autores conectados, autores no conectados o cercanía estructural.
   - TF-IDF: similitud textual o temática en Titulo, Keywords y Abstract.

4. Nota de alcance:
   - Aclara si el resultado es top-k, exploratorio o candidato preliminar.
   - En preguntas predictivas, aclara que no son colaboraciones garantizadas.

Redacta de forma clara, concreta y útil para un reporte de proyecto.
Evita texto genérico.
"""

    return prompt


# ============================================================
# Fallback sólo si se permite
# ============================================================

def respuesta_fallback(id_pregunta, pregunta, resultado):
    partes = [
        f"{id_pregunta}. {pregunta}",
        "",
        "Respuesta fallback generada a partir de resultados calculados:",
        resumen_resultado_para_llm(resultado)
    ]
    return "\n".join(partes)


# ============================================================
# Función final de redacción por pregunta
# ============================================================

def redactar_respuesta(id_pregunta, tipo, pregunta, componentes, resultado):
    """
    Genera la respuesta final.
    Si REQUERIR_LLM_EN_TODAS=True, se detiene si el LLM no responde.
    """

    prompt = construir_prompt_respuesta(
        id_pregunta=id_pregunta,
        tipo=tipo,
        pregunta=pregunta,
        componentes=componentes,
        resultado=resultado
    )

    respuesta = llamar_llm_ollama(prompt)

    if respuesta:
        return respuesta, True, prompt

    if REQUERIR_LLM_EN_TODAS:
        raise RuntimeError(
            f"El LLM local no generó respuesta para {id_pregunta}. "
            "La libreta está en modo estricto para comprobar uso de LLM en todas las preguntas."
        )

    return respuesta_fallback(id_pregunta, pregunta, resultado), False, prompt

Ollama detectado en: http://localhost:11434
Modelos disponibles:
 - qwen2.5:7b
LLM local disponible: qwen2.5:7b
Endpoint usado: http://localhost:11434/api/generate
Prueba inicial del modelo: OK


## 7. Ejecución de las 20 preguntas y comprobación de uso del LLM

In [7]:
respuestas = []
trazabilidad = []

for _, row in preguntas.iterrows():
    id_p = str(row["id"]).strip()
    tipo = str(row["tipo"]).strip()
    pregunta = str(row["pregunta"]).strip()
    componentes = str(row["componentes_consultados"]).strip()

    print(f"Resolviendo {id_p}: {pregunta}")

    if id_p not in RESOLUTORES:
        resultado = {
            "motores": componentes,
            "resultado": pd.DataFrame(),
            "nota": "No hay resolver definido para esta pregunta."
        }
    else:
        resultado = RESOLUTORES[id_p]()

    respuesta_final, uso_llm, prompt_usado = redactar_respuesta(id_p, tipo, pregunta, componentes, resultado)

    respuestas.append({
        "id": id_p,
        "tipo": tipo,
        "pregunta": pregunta,
        "componentes_consultados": componentes,
        "motores_usados": resultado.get("motores", componentes),
        "modelo_llm": MODELO_OLLAMA,
        "endpoint_llm": OLLAMA_URL,
        "uso_llm_local": uso_llm,
        "respuesta": respuesta_final
    })

    trazabilidad.append({
        "id": id_p,
        "tipo": tipo,
        "pregunta": pregunta,
        "componentes_consultados": componentes,
        "motores_usados": resultado.get("motores", componentes),
        "codigo_sql": resultado.get("codigo_sql", ""),
        "codigo_networkx": resultado.get("codigo_networkx", ""),
        "consulta_tfidf": resultado.get("consulta_tfidf", ""),
        "modelo_llm": MODELO_OLLAMA,
        "endpoint_llm": OLLAMA_URL,
        "uso_llm_local": uso_llm,
        "prompt_usado": prompt_usado[:12000],
        "resumen_resultado": resumen_resultado_para_llm(resultado)[:12000]
    })

respuestas_df = pd.DataFrame(respuestas)
trazabilidad_df = pd.DataFrame(trazabilidad)

# Comprobación estricta del uso de LLM.
num_preguntas = len(respuestas_df)
num_con_llm = int(respuestas_df["uso_llm_local"].sum())
print(f"Preguntas respondidas: {num_preguntas}")
print(f"Preguntas con LLM local: {num_con_llm}")

if REQUERIR_LLM_EN_TODAS and num_con_llm != num_preguntas:
    sin_llm = respuestas_df.loc[~respuestas_df["uso_llm_local"], "id"].tolist()
    raise RuntimeError(
        f"No todas las preguntas usaron el LLM local. Preguntas sin LLM: {sin_llm}. "
        "Revisa Ollama antes de entregar los resultados."
    )

print("Validación correcta: todas las preguntas usaron el LLM local.")
display(respuestas_df[["id", "tipo", "componentes_consultados", "motores_usados", "modelo_llm", "uso_llm_local"]])

Resolviendo D1: ¿Qué áreas de computación concentran más artículos por periodo de publicación?
Resolviendo D2: ¿Qué autores tienen mayor producción dentro de cada área de conocimiento?
Resolviendo D3: ¿Qué artículos tienen mayor número de coautores y a qué afiliaciones pertenecen esos autores?
Resolviendo D4: ¿Qué keywords aparecen con mayor frecuencia y en qué áreas se concentran?
Resolviendo D5: Para la keyword image processing, ¿qué afiliaciones concentran más autores y artículos relacionados con ese tema?
Resolviendo D6: ¿Qué autores conectan diferentes áreas o afiliaciones mediante sus publicaciones y coautorías?
Resolviendo D7: Para la keyword brain, ¿qué artículos la utilizan y qué autores participaron en ellos?
Resolviendo D8: ¿Qué áreas tienen producción reciente pero pocos autores asociados?
Resolviendo D9: ¿Qué autores trabajan temas similares, según keywords y abstracts, aunque todavía no aparezcan como coautores?
Resolviendo D10: ¿Qué publicaciones parecen duplicadas o var

,id,tipo,componentes_consultados,motores_usados,modelo_llm,uso_llm_local
0,D1,descriptiva,SQL + grafo principal,SQL + grafo principal,qwen2.5:7b,True
1,D2,descriptiva,SQL + grafo principal,SQL + grafo principal,qwen2.5:7b,True
2,D3,descriptiva,SQL + grafo principal,SQL + grafo principal,qwen2.5:7b,True
3,D4,descriptiva,SQL + TF-IDF + grafo principal,SQL + TF-IDF + grafo principal,qwen2.5:7b,True
4,D5,descriptiva,SQL + TF-IDF + grafo principal,SQL + TF-IDF + grafo principal,qwen2.5:7b,True
5,D6,descriptiva,SQL + grafo principal + subgrafo de coautoría,SQL + grafo principal + subgrafo de coautoría,qwen2.5:7b,True
6,D7,descriptiva,TF-IDF + grafo principal,TF-IDF + grafo principal,qwen2.5:7b,True
7,D8,descriptiva,SQL + grafo principal,SQL + grafo principal,qwen2.5:7b,True
8,D9,descriptiva,TF-IDF + grafo principal + subgrafo de coautoría,TF-IDF + grafo principal + subgrafo de coautoría,qwen2.5:7b,True
9,D10,descriptiva,SQL + TF-IDF,SQL + TF-IDF,qwen2.5:7b,True


## 8. Guardar respuestas, trazabilidad y diagnóstico

In [8]:
SALIDA_RESPUESTAS_CSV = DIR_CONSULTAS / "respuestas_20_preguntas_llm_corregido.csv"
SALIDA_TRAZABILIDAD_CSV = DIR_CONSULTAS / "trazabilidad_20_preguntas_llm_corregido.csv"
SALIDA_RESPUESTAS_TXT = DIR_CONSULTAS / "respuestas_20_preguntas_llm_corregido.txt"
SALIDA_DIAGNOSTICO_JSON = DIR_CONSULTAS / "diagnostico_uso_llm_corregido.json"
SALIDA_DIAGNOSTICO_TXT = DIR_CONSULTAS / "diagnostico_uso_llm_corregido.txt"

# Si estamos en fallback /mnt/data, guardar en esa misma carpeta de prueba.
if str(DIR_CONSULTAS).startswith("C:") and not DIR_CONSULTAS.exists():
    salida_dir = Path("/mnt/data")
    SALIDA_RESPUESTAS_CSV = salida_dir / "respuestas_20_preguntas_llm_corregido.csv"
    SALIDA_TRAZABILIDAD_CSV = salida_dir / "trazabilidad_20_preguntas_llm_corregido.csv"
    SALIDA_RESPUESTAS_TXT = salida_dir / "respuestas_20_preguntas_llm_corregido.txt"
    SALIDA_DIAGNOSTICO_JSON = salida_dir / "diagnostico_uso_llm_corregido.json"
    SALIDA_DIAGNOSTICO_TXT = salida_dir / "diagnostico_uso_llm_corregido.txt"

respuestas_df.to_csv(SALIDA_RESPUESTAS_CSV, index=False, encoding="utf-8-sig")
trazabilidad_df.to_csv(SALIDA_TRAZABILIDAD_CSV, index=False, encoding="utf-8-sig")

diagnostico = {
    "modelo_llm": MODELO_OLLAMA,
    "endpoint_llm": OLLAMA_URL,
    "ollama_detectado": bool(LLM_DISPONIBLE),
    "modelos_disponibles": LLM_INFO.get("modelos_disponibles", []),
    "preguntas_totales": int(len(respuestas_df)),
    "preguntas_con_llm": int(respuestas_df["uso_llm_local"].sum()),
    "todas_usaron_llm": bool(respuestas_df["uso_llm_local"].all()),
    "preguntas_sin_llm": respuestas_df.loc[~respuestas_df["uso_llm_local"], "id"].tolist(),
}

with open(SALIDA_DIAGNOSTICO_JSON, "w", encoding="utf-8") as f:
    json.dump(diagnostico, f, ensure_ascii=False, indent=2)

with open(SALIDA_RESPUESTAS_TXT, "w", encoding="utf-8") as f:
    f.write("RESPUESTAS A LAS 20 PREGUNTAS CON CONSULTAS HETEROGÉNEAS\n")
    f.write("=" * 72 + "\n\n")
    f.write(f"Modelo LLM local usado: {MODELO_OLLAMA}\n")
    f.write(f"Endpoint Ollama: {OLLAMA_URL}\n")
    f.write(f"Preguntas respondidas: {len(respuestas_df)}\n")
    f.write(f"Respuestas con LLM local activo: {int(respuestas_df['uso_llm_local'].sum())}\n")
    f.write(f"Todas usaron LLM local: {bool(respuestas_df['uso_llm_local'].all())}\n\n")
    for _, r in respuestas_df.iterrows():
        f.write("-" * 72 + "\n")
        f.write(f"{r['id']} | {r['tipo']} | {r['componentes_consultados']}\n")
        f.write(f"Motores usados: {r['motores_usados']}\n")
        f.write(f"Pregunta: {r['pregunta']}\n\n")
        f.write(str(r["respuesta"]).strip() + "\n\n")

with open(SALIDA_DIAGNOSTICO_TXT, "w", encoding="utf-8") as f:
    f.write("DIAGNÓSTICO DE USO DEL LLM LOCAL\n")
    f.write("=" * 72 + "\n\n")
    f.write(f"Modelo configurado: {MODELO_OLLAMA}\n")
    f.write(f"Endpoint usado: {OLLAMA_URL}\n")
    f.write(f"Ollama detectado: {LLM_DISPONIBLE}\n")
    f.write(f"Preguntas totales: {len(respuestas_df)}\n")
    f.write(f"Preguntas con LLM: {int(respuestas_df['uso_llm_local'].sum())}\n")
    f.write(f"Todas usaron LLM: {bool(respuestas_df['uso_llm_local'].all())}\n\n")
    f.write("Modelos disponibles detectados:\n")
    for m in LLM_INFO.get("modelos_disponibles", []):
        f.write(f"- {m}\n")
    if diagnostico["preguntas_sin_llm"]:
        f.write("\nPreguntas sin LLM:\n")
        for p in diagnostico["preguntas_sin_llm"]:
            f.write(f"- {p}\n")

print("Respuestas CSV:", SALIDA_RESPUESTAS_CSV)
print("Trazabilidad CSV:", SALIDA_TRAZABILIDAD_CSV)
print("Respuestas TXT:", SALIDA_RESPUESTAS_TXT)
print("Diagnóstico JSON:", SALIDA_DIAGNOSTICO_JSON)
print("Diagnóstico TXT:", SALIDA_DIAGNOSTICO_TXT)

Respuestas CSV: C:\Users\hazar\Desktop\PROYECTO\06_LLM\02_Consultas\respuestas_20_preguntas_llm_corregido.csv
Trazabilidad CSV: C:\Users\hazar\Desktop\PROYECTO\06_LLM\02_Consultas\trazabilidad_20_preguntas_llm_corregido.csv
Respuestas TXT: C:\Users\hazar\Desktop\PROYECTO\06_LLM\02_Consultas\respuestas_20_preguntas_llm_corregido.txt
Diagnóstico JSON: C:\Users\hazar\Desktop\PROYECTO\06_LLM\02_Consultas\diagnostico_uso_llm_corregido.json
Diagnóstico TXT: C:\Users\hazar\Desktop\PROYECTO\06_LLM\02_Consultas\diagnostico_uso_llm_corregido.txt


## 9. Vista rápida de respuestas

In [9]:
for _, r in respuestas_df.iterrows():
    print("=" * 100)
    print(f"{r['id']} | {r['tipo']} | {r['componentes_consultados']}")
    print(r["pregunta"])
    print("-" * 100)
    print(r["respuesta"][:3000])
    print()

D1 | descriptiva | SQL + grafo principal
¿Qué áreas de computación concentran más artículos por periodo de publicación?
----------------------------------------------------------------------------------------------------
1. Respuesta directa:
   Las áreas de computación que concentran más artículos en 2024 son IA (50 artículos) y SIAV (43 artículos). En 2025, estos roles se invierten con SIAV liderando (73 artículos).

2. Evidencia principal:
   |   Año | Area   |   articulos |
   |------:|:-------|------------:|
   |  2024 | IA     |          50 |
   |  2024 | SIAV   |          43 |
   |  2025 | SIAV   |          73 |
   |  2025 | IA     |          67 |

3. Criterio usado:
   SQL: conteos, agrupaciones o filtros.

4. Nota de alcance:
   El resultado es top-2 por año y se basa en la cantidad de artículos publicados en cada área durante los años 2024 y 2025.

D2 | descriptiva | SQL + grafo principal
¿Qué autores tienen mayor producción dentro de cada área de conocimiento?
--------------